# Fetch Database
This notebook downloads data from the Immich database for analysis:
- `photos.csv`: Metadata for all photos
- `embeddings.npy`: CLIP embeddings as a numpy array
- `embeddings_index.csv`: CSV mapping array indices to `assetId`

In [1]:
import sys
import os

# Add the src directory to path to import modules
sys.path.append(os.path.abspath('..'))

In [3]:
import numpy as np
import pandas as pd
import json
from src.db_client import get_all_photos, get_clip_embeddings

# Ensure data directory exists
os.makedirs('../data', exist_ok=True)

## 1. Download Photos Metadata

In [9]:
print("Fetching photos from database...")
photos_df = get_all_photos()
print(f"Fetched {len(photos_df)} photos.")

# Output memory usage and first few rows
photos_df.info()
display(photos_df.head())

photos_df.to_csv('../data/photos.csv', index=False)
print("Saved photos to ../data/photos.csv")

Fetching photos from database...
Fetched 6850 photos.
<class 'pandas.DataFrame'>
RangeIndex: 6850 entries, 0 to 6849
Data columns (total 24 columns):
 #   Column            Non-Null Count  Dtype              
---  ------            --------------  -----              
 0   id                6850 non-null   object             
 1   localDateTime     6850 non-null   datetime64[us, UTC]
 2   fileCreatedAt     6850 non-null   datetime64[us, UTC]
 3   isFavorite        6850 non-null   bool               
 4   visibility        6850 non-null   str                
 5   status            6850 non-null   str                
 6   width             6850 non-null   int64              
 7   height            6850 non-null   int64              
 8   stackId           0 non-null      object             
 9   latitude          6008 non-null   float64            
 10  longitude         6008 non-null   float64            
 11  city              6000 non-null   str                
 12  state             6

,id,localDateTime,fileCreatedAt,isFavorite,visibility,status,width,height,stackId,latitude,...,make,model,lensModel,focalLength,fNumber,iso,exposureTime,fileSizeInByte,dateTimeOriginal,rating
0,9528eb1e-c35c-4e47-a60a-85d4df0269a4,2021-05-13 17:52:28.310000+00:00,2021-05-13 17:52:28.310000+00:00,False,timeline,active,2250,4000,None,NaN,...,Xiaomi,M2003J15SC,NaN,4.74,1.8,100.0,1/328,3196654,2021-05-13 17:52:28.310000+00:00,NaN
1,2a2ea899-b59c-496f-8b30-5f09dd32fdf2,2021-05-13 17:52:28.310000+00:00,2021-05-13 17:52:28.310000+00:00,False,timeline,active,1127,1471,None,NaN,...,Xiaomi,M2003J15SC,NaN,4.74,1.8,100.0,1/328,404399,2021-05-13 17:52:28.310000+00:00,NaN
2,70b75c8f-d51d-4a90-8dd3-5e35164a76c0,2023-03-22 19:31:58+00:00,2023-03-22 19:31:58+00:00,False,timeline,active,1081,602,None,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,184813,2023-03-22 19:31:58+00:00,NaN
3,187f4121-6f57-4f9e-a579-6c266c656a64,2023-03-22 19:32:24+00:00,2023-03-22 19:32:24+00:00,False,timeline,active,1280,720,None,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,112162,2023-03-22 19:32:24+00:00,NaN
4,ab30abba-69c5-40e9-bd29-9a8107198618,2023-03-22 19:32:24+00:00,2023-03-22 19:32:24+00:00,False,timeline,active,720,1280,None,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,85045,2023-03-22 19:32:24+00:00,NaN


Saved photos to ../data/photos.csv


## 2. Download and Process Embeddings

In [6]:
print("Fetching embeddings from database...")
embeddings_df = get_clip_embeddings()
print(f"Fetched {len(embeddings_df)} embeddings.")

display(embeddings_df.head())

Fetching embeddings from database...
Fetched 6847 embeddings.


,assetId,embedding
0,c2dc8aee-b7cf-4cb7-8768-cddd43e2ca9c,"[-0.0034855632,0.00046971638,-0.004145324,0.02..."
1,4b349b6a-d798-40b5-9d14-64c99b3a4aee,"[-0.01783135,-0.01650325,0.0096772835,-0.01746..."
2,7e860d50-0641-4da5-b142-e3dc0b3c8dda,"[-0.038391072,0.014105344,0.014234288,-0.02303..."
3,721e4008-3f02-404d-9d8f-9faa0b584f37,"[0.043095987,0.01502027,-0.015513614,0.0376999..."
4,5b82a4a3-5a6a-47df-b1ef-481321a115cf,"[-0.006992431,0.030638607,0.02866625,0.0432699..."


In [7]:
print("Processing embeddings...")

embeddings_list = []
valid_indices = []

for idx, row in embeddings_df.iterrows():
    try:
        # Parse string representation (e.g., "[0.1, 0.2, ...]")
        emb = json.loads(row['embedding'])
        embeddings_list.append(emb)
        valid_indices.append(idx)
    except Exception as e:
        print(f"Error parsing embedding for assetId {row['assetId']}: {e}")

# Filter the dataframe to only include valid embeddings
valid_embeddings_df = embeddings_df.loc[valid_indices].copy()
valid_embeddings_df.reset_index(drop=True, inplace=True)

print(f"Successfully parsed {len(embeddings_list)} embeddings.")

Processing embeddings...
Successfully parsed 6847 embeddings.


In [8]:
# Save the embeddings as a numpy array
embeddings_array = np.array(embeddings_list, dtype=np.float32)
np.save('../data/embeddings.npy', embeddings_array)
print(f"Saved embeddings array of shape {embeddings_array.shape} to ../data/embeddings.npy")

# Save the index mapping array indices to assetId
index_df = valid_embeddings_df[['assetId']].copy()
index_df.to_csv('../data/embeddings_index.csv', index_label='index')
print("Saved embeddings index to ../data/embeddings_index.csv")

Saved embeddings array of shape (6847, 512) to ../data/embeddings.npy
Saved embeddings index to ../data/embeddings_index.csv
